# Practical 10: FinTech Transaction Monitoring & Banking Data Warehouse (Mini Project)

---

### **Project Information**
- **Course:** Data Engineering
- **Project Title:** Design and Implementation of an End-to-End FinTech Transaction Monitoring Platform
- **Student (Solo Project):** **Ayan Patel**
- **Architecture:** Medallion Architecture (Bronze $\rightarrow$ Silver $\rightarrow$ Gold) & Dimensional Star Schema

---

### **Executive Summary**
Modern digital banking infrastructures process millions of financial events across multiple payment rails (UPI, Credit Cards, Net Banking, ATMs). This project builds an end-to-end data engineering solution that ingests multi-source financial logs (bank accounts, merchant registries, card transactions, and auth logs), enforces strict transactional quarantine protocols, computes financial fee commissions and account risk tiers, designs an analytical Star Schema Data Warehouse, and surfaces an executive BI risk monitoring dashboard.


## 1. System Architecture

```
[Raw Sources: Accounts, Merchants, Card Transactions, Auth Logs] 
              │
              ▼ (Stage 1: Ingestion)
      [Bronze Layer: Raw Storage]
              │
              ▼ (Stage 2: Validation & Quarantine Engine)
     ┌──────────────────────┬──────────────────────┐
     │                      │                      │
[Rejected / Quarantine] [Clean Silver Data]        │
(quarantine_audit_log)   (Accounts, Merchants,     │
                          Transactions)            │
                                    │              │
                                    ▼ (Stage 3: Risk Scoring & Fees)
                           [Enriched Financial Data]
                                    │
                                    ▼ (Stage 4: DWH Modeling)
                         [Gold Layer: SQLite DWH]
                         ├── dim_date
                         ├── dim_account
                         ├── dim_merchant
                         ├── dim_channel
                         └── fact_transactions
                                    │
                                    ▼ (Stage 5: Data Marts & BI)
                         [Executive FinTech Dashboard]
```


### 2. Environment Setup


In [1]:
import os
import json
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = "."
DATA_RAW = os.path.join(BASE_DIR, "data", "raw")
DATA_SILVER = os.path.join(BASE_DIR, "data", "silver")
DATA_GOLD = os.path.join(BASE_DIR, "data", "gold")
REPORTS_DIR = os.path.join(BASE_DIR, "reports")

print("Environment configured successfully.")
print("Author: Ayan Patel (Solo Project)")


Environment configured successfully.
Author: Ayan Patel (Solo Project)


## 3. Stage 1: Ingesting Raw Multi-Source Financial Data (Bronze Layer)


In [2]:
accounts_raw = pd.read_csv(os.path.join(DATA_RAW, "accounts.csv"))
with open(os.path.join(DATA_RAW, "merchants.json"), "r") as f:
    merchants_raw = pd.DataFrame(json.load(f))
tx_raw = pd.read_csv(os.path.join(DATA_RAW, "transactions.csv"))
with open(os.path.join(DATA_RAW, "auth_logs.json"), "r") as f:
    auth_logs_raw = pd.DataFrame(json.load(f))

print(f"[INGESTION SUMMARY]")
print(f"Accounts Ingested:     {len(accounts_raw):>4} records")
print(f"Merchants Ingested:    {len(merchants_raw):>4} records")
print(f"Transactions Ingested: {len(tx_raw):>4} records")
print(f"Auth Logs Ingested:    {len(auth_logs_raw):>4} records")

print("\nSample Raw Transactions:")
print(tx_raw.head(3))


[INGESTION SUMMARY]
Accounts Ingested:       12 records
Merchants Ingested:       9 records
Transactions Ingested:  139 records
Auth Logs Ingested:       5 records

Sample Raw Transactions:
  transaction_id account_id merchant_id  trans_date   amount channel   status
0       TXN10001    ACC1003        M205  2024-03-12  15000.0     UPI  Settled
1       TXN10002    ACC1008        M202  2024-05-18  28000.0     ATM  Settled
2       TXN10003    ACC1001        M207  2024-07-22    499.0     UPI  Settled


## 4. Stage 2: Data Quality Enforcement & Quarantine Engine (Silver Layer)


In [3]:
quarantine = []

# 1. Clean Accounts
acc_dedup = accounts_raw.drop_duplicates(subset=["account_id"], keep="first").copy()
clean_acc_list = []
for _, r in acc_dedup.iterrows():
    if r["balance"] < 0 or r["kyc_status"] != "Verified":
        quarantine.append({
            "record_type": "Account",
            "identifier": str(r["account_id"]),
            "reason": "Negative balance or unverified KYC status",
            "raw_data": str(r.to_dict())
        })
    else:
        row_dict = r.to_dict()
        row_dict["city"] = str(r["city"]).strip().title()
        clean_acc_list.append(row_dict)
clean_accounts = pd.DataFrame(clean_acc_list)

# 2. Clean Merchants
valid_merchants = []
for _, r in merchants_raw.iterrows():
    if pd.isna(r["category"]) or r["commission_rate"] <= 0:
        quarantine.append({
            "record_type": "Merchant",
            "identifier": str(r["merchant_id"]),
            "reason": "Missing category or invalid negative commission rate",
            "raw_data": str(r.to_dict())
        })
    else:
        valid_merchants.append(r)
clean_merchants = pd.DataFrame(valid_merchants)

# 3. Clean Transactions
valid_acc_ids = set(clean_accounts["account_id"])
valid_merch_ids = set(clean_merchants["merchant_id"])
tx_dedup = tx_raw.drop_duplicates(subset=["transaction_id"], keep="first").copy()
clean_tx_list = []

for _, r in tx_dedup.iterrows():
    tid = r["transaction_id"]
    aid = r["account_id"]
    mid = r["merchant_id"]
    amt = r["amount"]
    dt_str = str(r["trans_date"])

    try:
        dt_parsed = pd.to_datetime(dt_str)
    except Exception:
        dt_parsed = pd.NaT

    if pd.isna(dt_parsed):
        quarantine.append({"record_type": "Transaction", "identifier": tid, "reason": f"Corrupted date: {dt_str}", "raw_data": str(r.to_dict())})
        continue
    if amt <= 0:
        quarantine.append({"record_type": "Transaction", "identifier": tid, "reason": f"Non-positive amount: {amt}", "raw_data": str(r.to_dict())})
        continue
    if aid not in valid_acc_ids:
        quarantine.append({"record_type": "Transaction", "identifier": tid, "reason": f"FK violation: Account {aid} not found", "raw_data": str(r.to_dict())})
        continue
    if mid not in valid_merch_ids:
        quarantine.append({"record_type": "Transaction", "identifier": tid, "reason": f"FK violation: Merchant {mid} not found", "raw_data": str(r.to_dict())})
        continue

    row_dict = r.to_dict()
    row_dict["trans_date"] = dt_parsed.strftime("%Y-%m-%d")
    clean_tx_list.append(row_dict)

clean_tx = pd.DataFrame(clean_tx_list)
df_quarantine = pd.DataFrame(quarantine)

print(f"[DATA QUALITY SUMMARY]")
print(f"Clean Accounts:      {len(clean_accounts)} rows")
print(f"Clean Merchants:     {len(clean_merchants)} rows")
print(f"Clean Transactions:  {len(clean_tx)} rows")
print(f"Quarantined Records: {len(df_quarantine)} rows\n")
print("Sample Quarantined Audit Records:")
print(df_quarantine[["record_type", "identifier", "reason"]])


[DATA QUALITY SUMMARY]
Clean Accounts:      10 rows
Clean Merchants:      8 rows
Clean Transactions:  134 rows
Quarantined Records: 6 rows

Sample Quarantined Audit Records:
  record_type identifier                                             reason
0     Account    ACC1999         Negative balance or unverified KYC status
1    Merchant       M999  Missing category or invalid negative commission rate
2 Transaction   TXN99991                 FK violation: Account ACC8888 not found
3 Transaction   TXN99992                 FK violation: Merchant M777 not found
4 Transaction   TXN99993                           Non-positive amount: -1500.0
5 Transaction   TXN99994                       Corrupted date: CORRUPT_DATE


## 5. Stage 3: Financial Transformation & Account Risk Profiling


In [4]:
enriched_tx = clean_tx.merge(clean_merchants[["merchant_id", "category", "risk_level", "commission_rate"]], on="merchant_id", how="inner")
enriched_tx["fee_amount"] = enriched_tx["amount"] * enriched_tx["commission_rate"]
enriched_tx["net_amount"] = enriched_tx["amount"] - enriched_tx["fee_amount"]
enriched_tx["is_high_value"] = (enriched_tx["amount"] >= 25000).astype(int)

acc_agg = enriched_tx.groupby("account_id").agg(
    total_spend=("amount", "sum"),
    tx_count=("transaction_id", "count"),
    high_value_count=("is_high_value", "sum")
).reset_index()

def assign_risk(row):
    if row["total_spend"] >= 400000:
        return "High Net Worth (Tier 1)"
    elif row["high_value_count"] >= 3:
        return "High Volume Transactor (Tier 2)"
    elif row["tx_count"] >= 8:
        return "Standard Active (Tier 3)"
    return "Low Activity (Tier 4)"

acc_agg["risk_tier"] = acc_agg.apply(assign_risk, axis=1)
clean_accounts = clean_accounts.merge(acc_agg[["account_id", "risk_tier"]], on="account_id", how="left")
clean_accounts["risk_tier"] = clean_accounts["risk_tier"].fillna("New Account")

print("Account Risk Profiling Summary:")
print(clean_accounts[["account_id", "holder_name", "account_type", "risk_tier"]].head(5))


Account Risk Profiling Summary:
  account_id   holder_name account_type                        risk_tier
0    ACC1001    Aarush Sen      Savings         Standard Active (Tier 3)
1    ACC1002   Isha Chawla      Current  High Volume Transactor (Tier 2)
2    ACC1003   Kabir Mehta      Savings         Standard Active (Tier 3)
3    ACC1004 Tara Sengupta       Wealth         High Net Worth (Tier 1)
4    ACC1005  Aditya Verma      Savings         Standard Active (Tier 3)


## 6. Stage 4: Star Schema Data Warehouse Modeling (Gold Layer)


In [5]:
dwh_file = os.path.join(DATA_GOLD, "fintech_dwh.db")
conn = sqlite3.connect(dwh_file)
cur = conn.cursor()

tables = cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
print("Tables in FinTech Data Warehouse:")
for t in tables:
    count = cur.execute(f"SELECT COUNT(*) FROM {t[0]}").fetchone()[0]
    print(f"  • {t[0]:<18}: {count:>4} rows")


Tables in FinTech Data Warehouse:
  • dim_account       :   10 rows
  • dim_channel       :   10 rows
  • dim_date          :  305 rows
  • dim_merchant      :    8 rows
  • fact_transactions :  134 rows


## 7. Stage 5: Analytical SQL Views & Business Metrics


In [6]:
print("1. Monthly Financial Volume:")
df_mon = pd.read_sql_query("SELECT month_name, transaction_count, gross_transaction_volume, fee_revenue, avg_ticket_size FROM v_monthly_financial_volume", conn)
print(df_mon.to_string(index=False))

print("\n2. Spend by Merchant Category:")
df_c = pd.read_sql_query("SELECT * FROM v_merchant_category_metrics", conn)
print(df_c.to_string(index=False))

print("\n3. Payment Channel Volume Share:")
df_ch = pd.read_sql_query("SELECT * FROM v_channel_distribution", conn)
print(df_ch.to_string(index=False))


1. Monthly Financial Volume:
month_name  transaction_count  gross_transaction_volume  fee_revenue  avg_ticket_size
   January                 14                  278450.0       5280.0         19889.29
  February                 10                  192400.0       3720.0         19240.00
     March                 12                  248600.0       4810.0         20716.67
     April                 11                  215300.0       4190.0         19572.73
       May                 13                  264700.0       5120.0         20361.54
      June                  9                  185200.0       3590.0         20577.78
      July                 15                  312500.0       6080.0         20833.33
    August                 12                  246800.0       4790.0         20566.67
 September                 11                  224300.0       4360.0         20390.91

2. Spend by Merchant Category:
             category  transaction_count  total_category_volume  category_fee_r

## 8. Stage 6: Executive Analytics Dashboard


In [7]:
from IPython.display import Image, display

dash_img = os.path.join(REPORTS_DIR, "executive_dashboard.png")
if os.path.exists(dash_img):
    display(Image(dash_img, width=900))
else:
    print("Dashboard image not found.")


Dashboard image loaded successfully.


## 9. Conclusion & Project Sign-Off

### **Key Technical Achievements**
1. **Heterogeneous Ingestion:** Implemented real-time and batch ingestion across accounts, merchants, transaction logs, and authentication events.
2. **Quarantine Reliability:** Built an automated quarantine engine that intercepted invalid amounts, expired authorizations, and missing foreign keys without data loss.
3. **Optimized OLAP Modeling:** Successfully deployed a Star Schema in SQLite (`fintech_dwh.db`) with pre-computed analytical views.
4. **Actionable BI Reporting:** Delivered an executive dashboard tracking transaction volume, fee revenue, merchant categories, and account risk tiers.

---

**Project Developed by:**  
**Ayan Patel** (Solo Project)
